In [1]:
!pip install pyiceberg[s3fs,pandas,pyarrow] boto3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.6/842.6 kB 11.3 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 18.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: pi

In [2]:
import os
from pyiceberg.catalog import load_catalog
import pandas as pd
import pyarrow as pa
from datetime import datetime

os.environ['PYICEBERG_DOWNCAST_NS_TIMESTAMP_TO_US_ON_WRITE'] = 'true'

In [3]:
# Connect to Iceberg REST Catalog
catalog = load_catalog(
    "rest",
    **{
        "uri": "http://iceberg-rest:8181",
        "s3.endpoint": "http://minio:9000",
        "s3.access-key-id": "admin",
        "s3.secret-access-key": "Password!",
        "s3.path-style-access": "true",
        "s3.region": "us-east-1" 
    }
)

print("Connected to Iceberg REST Catalog!")
print(f"Catalog properties: {catalog.properties}")

Connected to Iceberg REST Catalog!
Catalog properties: {'uri': 'http://iceberg-rest:8181', 's3.endpoint': 'http://minio:9000', 's3.access-key-id': 'admin', 's3.secret-access-key': 'Password!', 's3.path-style-access': 'true', 's3.region': 'us-east-1'}


In [4]:
# Create Namespace (Database)
try:
    catalog.create_namespace("demo")
    print("Created namespace: demo")
except Exception as e:
    print(f"Namespace may already exist: {e}")

# List namespaces
print("\nAvailable namespaces:", catalog.list_namespaces())

Created namespace: demo

Available namespaces: [('demo',)]


In [5]:
# Drop existing table
try:
    catalog.drop_table("demo.events")
    print("Dropped existing table")
except:
    pass

In [6]:
# Create Table Schema
from pyiceberg.schema import Schema
from pyiceberg.types import (
    NestedField,
    StringType,
    DoubleType,
    TimestampType,
    LongType  # ← Changed from IntegerType
)

schema = Schema(
    NestedField(1, "id", LongType(), required=False),  # ← Changed to LongType and optional
    NestedField(2, "timestamp", TimestampType(), required=False),  # ← Made optional
    NestedField(3, "user_id", StringType(), required=False),  # ← Made optional
    NestedField(4, "event_type", StringType(), required=False),  # ← Made optional
    NestedField(5, "value", DoubleType(), required=False),
)

# Create table
try:
    table = catalog.create_table(
        identifier="demo.events",
        schema=schema,
    )
    print("Created table: demo.events")
except Exception as e:
    print(f"Table may already exist: {e}")
    table = catalog.load_table("demo.events")

print(f"\nTable schema:\n{table.schema()}")

Created table: demo.events

Table schema:
table {
  1: id: optional long
  2: timestamp: optional timestamp
  3: user_id: optional string
  4: event_type: optional string
  5: value: optional double
}


In [7]:
# Write Data

data = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "timestamp": pd.date_range("2024-01-01", periods=5, freq="H"),
    "user_id": ["user_1", "user_2", "user_1", "user_3", "user_2"],
    "event_type": ["login", "click", "purchase", "login", "click"],
    "value": [None, 10.5, 99.99, None, 25.0]
})

# Convert timestamp to microsecond precision
data['timestamp'] = data['timestamp'].astype('datetime64[us]')

# Convert to PyArrow table
arrow_table = pa.Table.from_pandas(data)

print("Writing data to Iceberg table...")
table.append(arrow_table)
print("✓ Data written successfully!")

Writing data to Iceberg table...
✓ Data written successfully!


In [19]:
#  Read Data

from pyiceberg.table import TableProperties
table = catalog.load_table("demo.events")

with table.transaction() as txn:
    txn.set_properties(
        **{TableProperties.DEFAULT_NAME_MAPPING: table.metadata.schema().name_mapping.model_dump_json()}
    )

print(f"name-mapping: {table.metadata.name_mapping()}")

print("Reading data from table...")
df = table.scan().to_pandas()
print(f"\nTable has {len(df)} rows:\n")
print(df)

name-mapping: [
  ([id] -> 1)
  ([timestamp] -> 2)
  ([user_id] -> 3)
  ([event_type] -> 4)
  ([value] -> 5)
]
Reading data from table...


ResolveError: Cannot promote timestamptz to timestamp

In [9]:
# Query with Filters
print("\n--- Filtering: event_type = 'login' ---")
df_filtered = table.scan(
    row_filter="event_type == 'login'"
).to_pandas()
print(df_filtered)

print("\n--- Filtering: value > 20 ---")
df_filtered2 = table.scan(
    row_filter="value > 20"
).to_pandas()
print(df_filtered2)


--- Filtering: event_type = 'login' ---
   id           timestamp user_id event_type  value
0   1 2024-01-01 00:00:00  user_1      login    NaN
1   4 2024-01-01 03:00:00  user_3      login    NaN

--- Filtering: value > 20 ---
   id           timestamp user_id event_type  value
0   3 2024-01-01 02:00:00  user_1   purchase  99.99
1   5 2024-01-01 04:00:00  user_2      click  25.00


In [10]:
# Append More Data
new_data = pd.DataFrame({
    "id": [6, 7, 8],
    "timestamp": pd.date_range("2024-01-01 05:00:00", periods=3, freq="H"),
    "user_id": ["user_1", "user_4", "user_2"],
    "event_type": ["logout", "login", "purchase"],
    "value": [None, None, 149.99]
})

# Convert timestamp to microsecond precision (Iceberg requirement)
new_data['timestamp'] = new_data['timestamp'].astype('datetime64[us]')

# Convert to PyArrow table
arrow_new_data = pa.Table.from_pandas(new_data)

print("Appending more data...")
table.append(arrow_new_data)
print("✓ Data appended!")

# Read updated data
df_updated = table.scan().to_pandas()
print(f"\nTable now has {len(df_updated)} rows")
print(df_updated)

Appending more data...
✓ Data appended!

Table now has 8 rows
   id           timestamp user_id event_type   value
0   6 2024-01-01 05:00:00  user_1     logout     NaN
1   7 2024-01-01 06:00:00  user_4      login     NaN
2   8 2024-01-01 07:00:00  user_2   purchase  149.99
3   1 2024-01-01 00:00:00  user_1      login     NaN
4   2 2024-01-01 01:00:00  user_2      click   10.50
5   3 2024-01-01 02:00:00  user_1   purchase   99.99
6   4 2024-01-01 03:00:00  user_3      login     NaN
7   5 2024-01-01 04:00:00  user_2      click   25.00


In [11]:
# Table History & Time Travel
print("--- Table History ---")
snapshots = table.metadata.snapshots
for snapshot in snapshots:
    print(f"Snapshot ID: {snapshot.snapshot_id}, Timestamp: {snapshot.timestamp_ms}")

# Time travel - read data as of first snapshot
if len(snapshots) >= 2:
    first_snapshot_id = snapshots[0].snapshot_id
    print(f"\n--- Time Travel to Snapshot {first_snapshot_id} ---")
    df_historical = table.scan(snapshot_id=first_snapshot_id).to_pandas()
    print(f"Historical data ({len(df_historical)} rows):")
    print(df_historical)

--- Table History ---
Snapshot ID: 3943851647828089965, Timestamp: 1766711596326
Snapshot ID: 4019246713139795997, Timestamp: 1766711606759

--- Time Travel to Snapshot 3943851647828089965 ---
Historical data (5 rows):
   id           timestamp user_id event_type  value
0   1 2024-01-01 00:00:00  user_1      login    NaN
1   2 2024-01-01 01:00:00  user_2      click  10.50
2   3 2024-01-01 02:00:00  user_1   purchase  99.99
3   4 2024-01-01 03:00:00  user_3      login    NaN
4   5 2024-01-01 04:00:00  user_2      click  25.00
